# 04. Invoking the CloudXeus Invoice Agent with Content-Understanding-Extracted Data

This is the pipeline's "run" step, and it chains **three** separate Azure AI services together:

1. **Azure AI Content Understanding** runs a *custom, schema-driven* analyzer (`cloudxeusstudentinvoiceanalyzer`) against a sample CloudXeus invoice PDF, extracting structured fields (student name, invoice number, course code, total amount, payment status, ...) as defined by the field schema in the sibling file `cloudxeus-student-invoice-analyzer.json`.
2. The extracted structured data is embedded into a prompt.
3. `project_client.get_openai_client()` bridges Foundry's Agent Service into the familiar OpenAI-compatible **Responses API**, and the `extra_body.agent_reference` parameter routes the request through the **persistent agent created in `03_cloudxeus_invoice_agent.ipynb`** — looked up by the exact same name, `AGENT_NAME = "CloudXeus-Invoice-Intelligence-Agent"`.

**This notebook has a hard dependency on the two notebooks before it.** `03_cloudxeus_invoice_agent.ipynb` must have already run successfully (so an agent named `CloudXeus-Invoice-Intelligence-Agent` exists in the project), and — transitively — `02_project_mcp_conn.ipynb` must have already run too (so that agent's MCP tool has a live connection to call when it needs product information). If the agent doesn't exist yet, `responses.create(...)` with this `agent_reference` will fail.

**Difficulty: Advanced**

## Prerequisites

**Python packages**
```bash
pip install azure-ai-contentunderstanding azure-ai-projects azure-identity python-dotenv
```
`azure-ai-contentunderstanding` is a newer/preview package (following the same `azure.ai.<service>` → `azure-ai-<service>` naming convention as `azure-ai-documentintelligence`) and is **not** in the repo root `requirements.txt` — install it explicitly, and check PyPI / the Azure AI Foundry SDK docs for the current package name and version if the import fails, since preview SDKs move fast.

**Azure resources**
- An **Azure AI Content Understanding** resource (this script's `CONTENT_UNDERSTANDING_ENDPOINT` points at the *same* Foundry account host used for the project endpoint — Content Understanding is exposed through the Foundry account).
- A **custom Content Understanding analyzer** already created from the schema in `../cloudxeus-student-invoice-analyzer.json` (built on `baseAnalyzerId: "prebuilt-document"`, extracting `studentName`, `studentEmail`, `invoiceNumber`, `invoiceDate`, `courseCode`, `courseName`, `totalAmount`, `paymentStatus`).
- The Foundry project + agent from `03_cloudxeus_invoice_agent.ipynb`.

⚠️ **Naming gotcha:** the JSON schema file's `analyzerId` is `"cloudxeus-student-invoice-analyzer"` (with hyphens), but the original script's `ANALYZER_ID` constant is `"cloudxeusstudentinvoiceanalyzer"` (no hyphens). Whichever ID you actually provisioned the analyzer under in your Content Understanding resource is the one that must go in `AZURE_CU_ANALYZER_ID` below — a mismatch here is a very real, very easy source of a 404.

**Environment variables** (add to the root `.env`):

```
AZURE_CONTENT_UNDERSTANDING_ENDPOINT=https://<your-foundry-account>.services.ai.azure.com/
AZURE_CONTENT_UNDERSTANDING_KEY=
AZURE_CU_ANALYZER_ID=cloudxeusstudentinvoiceanalyzer
INVOICE_BLOB_URL=https://<your-storage-account>.blob.core.windows.net/.../CloudXeus_Invoice_INV-CX-2026-1003.pdf
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-account>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_AGENT_NAME=CloudXeus-Invoice-Intelligence-Agent
```

Leave `AZURE_CONTENT_UNDERSTANDING_KEY` blank to authenticate with `DefaultAzureCredential` instead — the code below preserves the original script's fallback logic exactly. The sample file referenced here, `CloudXeus_Invoice_INV-CX-2026-1003.pdf`, also has a local copy in `../cloudxeus_invoices/`.

## What You'll Learn

- How **Azure AI Content Understanding** differs from Document Intelligence: it's schema-driven, defined by a JSON field schema (see `../cloudxeus-student-invoice-analyzer.json`), and returns typed, named fields rather than generic layout
- The same `begin_analyze` → `poller.result()` long-running-operation pattern from notebook 1, applied to a different service
- How to fall back between key-based and Entra ID auth in one block (`AzureKeyCredential` vs `DefaultAzureCredential`)
- `project_client.get_openai_client()` — bridging the Foundry Agent Service into the OpenAI Responses API surface
- The `extra_body={"agent_reference": ...}` pattern for invoking a named, persistent, tool-equipped agent instead of a bare chat model

### Imports and environment-driven configuration

Every hardcoded endpoint, key, analyzer ID, and blob URL becomes an environment variable. Note that `CONTENT_UNDERSTANDING_KEY` is deliberately allowed to be empty — the next cell branches on that.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisInput
from azure.ai.projects import AIProjectClient
from dotenv import load_dotenv

load_dotenv()

CONTENT_UNDERSTANDING_ENDPOINT = os.getenv("AZURE_CONTENT_UNDERSTANDING_ENDPOINT")
CONTENT_UNDERSTANDING_KEY = os.getenv("AZURE_CONTENT_UNDERSTANDING_KEY", "")

ANALYZER_ID = os.getenv("AZURE_CU_ANALYZER_ID", "cloudxeusstudentinvoiceanalyzer")

# Sample CloudXeus invoice PDF. Local copy: ../cloudxeus_invoices/CloudXeus_Invoice_INV-CX-2026-1003.pdf
INVOICE_BLOB_URL = os.getenv(
    "INVOICE_BLOB_URL",
    "https://stcxai103capdeus01.blob.core.windows.net/course-products/student-invoices/CloudXeus_Invoice_INV-CX-2026-1003.pdf",
)

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "CloudXeus-Invoice-Intelligence-Agent")


### Resolving credentials and creating the Content Understanding client

This ternary preserves the original script's fallback exactly: if a key is present in the environment, use simple key-based auth (`AzureKeyCredential`); otherwise fall back to `DefaultAzureCredential` (Entra ID, e.g. your `az login` session or a managed identity in production).

💡 **Exam tip (AI-102):** this "key if present, otherwise Entra ID" pattern is a common, exam-relevant way to write code that works identically for a developer's local key-based resource and a production deployment secured with managed identity and no keys at all — no code branching needed beyond this one `if`.

In [ ]:
credential = (
    AzureKeyCredential(CONTENT_UNDERSTANDING_KEY)
    if CONTENT_UNDERSTANDING_KEY
    else DefaultAzureCredential()
)

content_client = ContentUnderstandingClient(
    endpoint=CONTENT_UNDERSTANDING_ENDPOINT,
    credential=credential,
)


### Running the custom analyzer

`begin_analyze` is another long-running operation, just like `begin_analyze_document` in notebook 1 — but here it's driven by `analyzer_id`, a **custom** analyzer definition (the JSON schema in `../cloudxeus-student-invoice-analyzer.json`) rather than a generic prebuilt model. The result contains the schema's named fields (`studentName`, `invoiceNumber`, `courseCode`, `totalAmount`, `paymentStatus`, ...) instead of raw layout/markdown.

💡 **Exam tip (AI-102):** Content Understanding analyzers are built from a `fieldSchema` on top of a `baseAnalyzerId` (here, `prebuilt-document`), and can be configured to `estimateFieldSourceAndConfidence` — returning, per field, not just the extracted value but a confidence score and where in the document it was sourced from. This is the "custom extraction model" AI-102 candidates should be able to contrast against Document Intelligence's generic `prebuilt-layout`/`prebuilt-invoice` models used in notebook 1.

🔄 **Alternatives:** If your fields map closely to a standard invoice (vendor, total, due date, line items) and you don't need custom fields like `studentName` or `courseCode`, Document Intelligence's `prebuilt-invoice` model (see notebook 1's exercises) may be sufficient and requires no analyzer authoring step at all.

In [ ]:
poller = content_client.begin_analyze(
    analyzer_id=ANALYZER_ID,
    inputs=[
        AnalysisInput(url=INVOICE_BLOB_URL)
    ],
)

invoice_result = poller.result()

print("Invoice analysis completed.")


### Building the prompt and invoking the CloudXeus agent

`project_client.get_openai_client()` returns an OpenAI-SDK-compatible client backed by the Foundry project — so calls like `.responses.create(...)` use the familiar OpenAI Responses API shape. The key detail is `extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}}`: this tells Foundry to route the request through the **named, persistent agent** created in `03_cloudxeus_invoice_agent.ipynb` — with its full system instructions and MCP knowledge-base tool — rather than hitting a bare chat model directly. The prompt embeds the just-extracted, structured invoice JSON (`invoice_result.as_dict()`) and asks for a seven-part intelligence report spanning finance, sales, and customer-facing angles, exactly matching the instructions the agent was given in notebook 3.

💡 **Exam tip (AI-102):** `agent_reference` is Foundry's mechanism for letting an otherwise-standard OpenAI-Responses-API call be handled by a persistent, tool-equipped agent instead of a stateless model call — all of that agent's system instructions, tool wiring, and `tool_choice` policy from its definition apply automatically, without the caller having to re-specify any of it here.

🔄 **Alternatives:** Instead of the OpenAI Responses `agent_reference` shortcut, you could drive the same agent using the `azure-ai-agents` SDK's thread-based pattern (`agents_client.threads.create()` → `messages.create()` → `runs.create_and_process()`), as shown in `11_azure_ai_foundry/05_hosted_agent/`. That pattern also gives you a persistent **thread** with conversation memory across multiple calls, which this one-shot `responses.create` call does not.

In [ ]:
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

openai_client = project_client.get_openai_client()

prompt = f"""
Create a complete invoice intelligence report.

Use the extracted invoice data below.
When you need product information, use the CloudXeus product knowledge base.

Include:
1. Invoice overview
2. Products purchased
3. Product explanations from the knowledge base
4. Product match validation
5. Finance summary
6. Customer-friendly explanation
7. Recommended sales follow-up

Extracted invoice data:
{json.dumps(invoice_result.as_dict(), indent=2)}
"""

response = openai_client.responses.create(
    extra_body={
        "agent_reference": {
            "name": AGENT_NAME,
            "type": "agent_reference"
        }
    },
    input=prompt,
)

print("\n----- Agent Report -----\n")
print(response.output_text)


## Summary

We ran a custom Content Understanding analyzer against a sample CloudXeus invoice, got back structured fields (not raw text), serialized them into a prompt, and invoked the `CloudXeus-Invoice-Intelligence-Agent` created in `03_cloudxeus_invoice_agent.ipynb` via the OpenAI Responses API's `agent_reference` mechanism. The printed `response.output_text` is the agent's seven-part intelligence report — during generation, the agent may have transparently called its MCP knowledge-base tool one or more times (per its `tool_choice="auto"` setting) to look up CloudXeus product details before answering. This notebook is the payoff of the whole section's pipeline: extraction (Content Understanding) feeding a grounded, tool-using agent (Foundry Agent Service + MCP).

## Try It Yourself

1. **Easy:** Point `INVOICE_BLOB_URL` at a different sample invoice in `../cloudxeus_invoices/` and compare the generated report — does the agent flag a different set of products or amounts?
2. **Intermediate:** Print `invoice_result.as_dict()` on its own, before building the prompt, and inspect the raw field-level confidence scores Content Understanding returned (if `estimateFieldSourceAndConfidence` is enabled on your analyzer) — add a check that warns if any field's confidence is below a threshold, before trusting it in the prompt.
3. **Advanced:** Rewrite the agent invocation to use the `azure-ai-agents` thread-based pattern from `11_azure_ai_foundry/05_hosted_agent/` instead of `agent_reference`, so you can ask a *follow-up* question in the same thread (e.g. "What was the tax amount again?") and confirm the agent remembers the first invoice without you re-sending the extracted data.